# 🧠 Tutorial 01: Introducción al Meta-Learning

## "Aprender a Aprender"

Bienvenido al primer tutorial de Meta-Learning! En este notebook aprenderás:

- 📚 Qué es Meta-Learning y por qué es importante
- 🔄 Diferencia entre ML tradicional, Transfer Learning y Meta-Learning
- 🎯 El concepto fundamental de "conjunto de tareas" vs "conjunto de datos"
- 💻 Tu primera implementación práctica de Few-Shot Learning

---

## 📖 Parte 1: Teoría

### ¿Qué es Meta-Learning?

**Meta-Learning** (o "aprender a aprender") es un paradigma de Machine Learning donde el objetivo no es simplemente optimizar parámetros para una tarea específica, sino **optimizar el proceso de aprendizaje mismo**.

#### Analogía del Estudiante:

- **ML Tradicional**: Es como un estudiante que memoriza las respuestas de un examen específico.
- **Transfer Learning**: Es como un estudiante que usa conocimiento de una materia similar.
- **Meta-Learning**: Es como un estudiante que aprende **técnicas de estudio** que le permiten dominar cualquier nueva materia rápidamente.

### Conceptos Clave:

1. **Conjunto de Tareas (Task Set)**: En lugar de entrenar con un dataset, entrenamos con múltiples tareas relacionadas.

2. **Support Set (Soporte)**: Pequeño conjunto de ejemplos para adaptar el modelo a una nueva tarea.

3. **Query Set (Consulta)**: Conjunto de test para evaluar la adaptación.

4. **Few-Shot Learning**: Aprender de pocos ejemplos (1-shot, 5-shot, etc.)

### La Gran Diferencia:

```
ML Tradicional:
  Datos (muchos) → Modelo → Predicciones en la misma distribución

Meta-Learning:
  Múltiples Tareas → Meta-Modelo → Adaptación rápida a NUEVAS tareas
```


## 📑 Table of Contents- [1 - What is Meta-Learning?](#1)    - [1.1 - The Core Concept](#1-1)    - [1.2 - Real-World Motivation](#1-2)    - [1.3 - Key Terminology](#1-3)- [2 - Why Meta-Learning Matters](#2)- [3 - Three Main Paradigms](#3)    - [3.1 - Metric-Based](#3-1)    - [3.2 - Model-Based](#3-2)    - [3.3 - Optimization-Based](#3-3)- [4 - Few-Shot Learning](#4)- [5 - Key Algorithms Overview](#5)- [6 - Exercise: Identify the Paradigm](#ex-1)- [7 - Success Stories](#7)- [8 - Challenges and Limitations](#8)- [9 - Course Roadmap](#9)- [10 - Getting Started](#10)

---

## 🛠️ Parte 2: Setup y Librerías

In [ ]:
# Importar librerías necesarias
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from utils.test_utils import (
    print_success, print_hint, HintSystem, 
    check_implementation, run_test
)
from utils.data_utils import create_sine_task, set_seed
from utils.visualization import plot_few_shot_results

# Configurar semilla para reproducibilidad
set_seed(42)

# Verificar si GPU está disponible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Usando dispositivo: {device}")
print("✅ Librerías importadas correctamente!")

<a name='1-2'></a>### 1.2 - Real-World Motivation: Why Do We Need Meta-Learning?Traditional machine learning has a fundamental limitation:<table><tr>    <td><b>Scenario</b></td>    <td><b>Traditional ML</b></td>    <td><b>Human Learning</b></td></tr><tr>    <td>Learning to recognize cats</td>    <td>Needs 1000s of cat images</td>    <td>Sees 3-5 cats, learns the concept</td></tr><tr>    <td>Learning a new game</td>    <td>Millions of games (AlphaGo)</td>    <td>Few examples, reads rules once</td></tr><tr>    <td>Medical diagnosis</td>    <td>Thousands of labeled cases</td>    <td>Doctors learn from handful of cases</td></tr><tr>    <td>Adapting to new domain</td>    <td>Retrain from scratch</td>    <td>Transfer knowledge quickly</td></tr></table>**The Gap**: Traditional ML needs massive data. Humans learn from very few examples.**Meta-Learning's Promise**: Bridge this gap by learning HOW to learn efficiently.### Real Applications:1. **Drug Discovery**: Test new molecules with limited experiments2. **Personalized Medicine**: Adapt to individual patients (limited data per person)3. **Robotics**: Learn new tasks quickly without millions of trials4. **Few-Shot Image Recognition**: Recognize new species/objects from few photos5. **Neural Architecture Search**: Learn to design better neural networks

<a name='1-3'></a>### 1.3 - Key Terminology (Essential Vocabulary)Understanding these terms is crucial for this course:**Task (T)**: A complete learning problem- Example: "Classify images of dogs vs cats"- Contains: Dataset, objective, evaluation metric**Task Distribution (p(T))**: A distribution over tasks- Example: "Classify animals" (tasks: dogs/cats, birds/fish, etc.)- Meta-learning assumes tasks come from a shared distribution**Support Set (S)**: Training data for a task- Also called: "Training examples", "Context examples"- In 5-way 1-shot: 5 classes × 1 example = 5 images**Query Set (Q)**: Test data for a task- Also called: "Test examples", "Target examples"- Used to evaluate how well model adapted to the task**N-way K-shot**: Standard problem formulation- **N-way**: Number of classes (e.g., 5 classes)- **K-shot**: Examples per class (e.g., 1 example per class)- Example: "5-way 1-shot" = classify among 5 classes using 1 example each**Episode**: One meta-training iteration- Sample a task from p(T)- Adapt using support set- Evaluate on query set- Update meta-parameters**Meta-Training**: Learning across many tasks- Goal: Learn to learn from the task distribution**Meta-Testing**: Evaluating on new tasks- Uses tasks never seen during meta-training- Tests if model learned to learn (not just memorized)### Visual Example:```Meta-Training:  Episode 1: Task = "Dogs vs Cats"    Support: [🐕, 🐈]    Query:   [🐕?, 🐈?, 🐕?]      Episode 2: Task = "Birds vs Fish"    Support: [🐦, 🐟]    Query:   [🐦?, 🐟?, 🐦?]      ... (1000s of episodes)  Meta-Testing:  New Task: "Horses vs Cows" (never seen before!)    Support: [🐴, 🐄]    Query:   [🐴?, 🐄?, 🐴?]      Can the model classify correctly? → Test of meta-learning!```

---

## 📊 Parte 3: Visualizando el Problema

Vamos a crear nuestro primer ejemplo de Meta-Learning: **regresión de funciones sinusoidales**.

Cada tarea será aprender una función seno con amplitud y fase diferentes:
$$y = A \sin(x + \phi)$$

In [ ]:
# Generar una tarea de ejemplo
task = create_sine_task(k_shot=10, q_query=50)

print(f"📊 Tarea generada:")
print(f"  Amplitud: {task['amplitude']:.2f}")
print(f"  Fase: {task['phase']:.2f}")
print(f"  Support set: {task['x_support'].shape}")
print(f"  Query set: {task['x_query'].shape}")

# Visualizar
plot_few_shot_results(
    task['x_support'], task['y_support'],
    task['x_query'], task['y_query'],
    title="Ejemplo de Tarea de Meta-Learning: Regresión Sinusoidal"
)

<a name='5'></a>## 5 - Key Algorithms OverviewHere are the main algorithms you'll learn in this course:### Metric-Based Methods:**1. Prototypical Networks** (Tutorial 03)- **Idea**: Represent each class by its prototype (centroid)- **Classification**: Find nearest prototype- **Pros**: Simple, fast, interpretable- **Cons**: Assumes Euclidean space is meaningful**2. Matching Networks** (Tutorial 03c)- **Idea**: Attention over all support examples- **Classification**: Weighted vote using attention- **Pros**: Flexible, interpretable attention- **Cons**: More complex than Prototypical**3. Relation Networks**- **Idea**: Learn the similarity metric- **Classification**: Neural network computes relation score- **Pros**: Learns optimal metric for task- **Cons**: More parameters to learn### Optimization-Based Methods:**4. MAML** (Tutorial 04)- **Idea**: Learn initialization for fast adaptation- **Method**: Bi-level optimization (inner + outer loop)- **Pros**: Model-agnostic, strong performance- **Cons**: Expensive (second-order gradients)**5. Reptile**- **Idea**: Simplified MAML- **Method**: First-order approximation- **Pros**: Faster than MAML- **Cons**: Slightly worse performance### Model-Based Methods:**6. Memory-Augmented Networks** (Tutorial 05)- **Idea**: External memory for fast adaptation- **Method**: LSTM/NTM with memory- **Pros**: Can store task-specific info- **Cons**: Complex architecture### Performance Comparison (Mini-ImageNet 5-way 1-shot):<table><tr>    <td><b>Method</b></td>    <td><b>Accuracy</b></td>    <td><b>Training Speed</b></td>    <td><b>Inference Speed</b></td></tr><tr>    <td>Matching Networks</td>    <td>43.6%</td>    <td>Fast</td>    <td>Fast</td></tr><tr>    <td>Prototypical Networks</td>    <td>49.4%</td>    <td>Fast</td>    <td>Very Fast</td></tr><tr>    <td>Relation Networks</td>    <td>50.4%</td>    <td>Medium</td>    <td>Medium</td></tr><tr>    <td>MAML</td>    <td>48.7%</td>    <td>Slow</td>    <td>Medium</td></tr></table>

### 🤔 Observación Importante:

Nota que tenemos **solo 10 ejemplos de soporte** (puntos azules). 

- ❌ **ML Tradicional**: Necesitaría cientos o miles de puntos para aprender bien
- ✅ **Meta-Learning**: Puede adaptarse con muy pocos ejemplos si ha visto tareas similares antes

<a name='7'></a>## 7 - Success Stories: Meta-Learning in Action### 1. **Medical Imaging** (Stanford University)- **Problem**: Rare disease diagnosis (limited labeled data)- **Solution**: Prototypical Networks for few-shot medical image classification- **Result**: 15% improvement over transfer learning with <100 examples### 2. **Drug Discovery** (DeepMind)- **Problem**: Predict molecular properties (expensive experiments)- **Solution**: MAML for few-shot molecular property prediction- **Result**: Accurate predictions with 10x fewer experiments### 3. **Robotics** (UC Berkeley)- **Problem**: Robot needs to learn new tasks quickly- **Solution**: Model-Agnostic Meta-Learning (MAML)- **Result**: Robot learns new manipulation tasks from 10-20 demonstrations### 4. **Neural Architecture Search** (Google Brain)- **Problem**: Finding optimal network architectures- **Solution**: Meta-learning the architecture search strategy- **Result**: Found architectures competitive with human-designed ones### 5. **Personalized Recommendations**- **Problem**: New user with no history ("cold start")- **Solution**: Meta-learn from other users' patterns- **Result**: Better recommendations from first few interactions### 6. **Low-Resource Language Translation**- **Problem**: Translate rare languages (limited parallel text)- **Solution**: Meta-learning across high-resource language pairs- **Result**: Decent translation with 1000s instead of millions of sentences

<a name='8'></a>## 8 - Challenges and LimitationsMeta-learning is powerful but not a silver bullet. Current challenges:### 1. **Task Distribution Mismatch**- **Problem**: Meta-test tasks must be similar to meta-train tasks- **Reality**: If test tasks are too different, performance degrades- **Example**: Meta-trained on animals, tested on vehicles → poor results### 2. **Computational Cost**- **Problem**: MAML requires expensive second-order gradients- **Impact**: 10-100x slower than standard training- **Mitigation**: First-order approximations (FOMAML, Reptile)### 3. **Limited Theoretical Understanding**- **Problem**: Why does meta-learning work? When will it fail?- **Status**: Active research area, limited theoretical guarantees- **Progress**: Recent PAC-learning bounds, generalization theory### 4. **Scalability Issues**- **Problem**: Meta-training requires many tasks- **Requirement**: 1000s-10000s of tasks for good performance- **Limitation**: Not always feasible to collect so many tasks### 5. **Hyperparameter Sensitivity**- **Problem**: Performance sensitive to learning rates, architecture choices- **Example**: Inner vs outer learning rates in MAML- **Solution**: Careful tuning, sometimes automated hyperparameter search### 6. **Overfitting to Task Distribution**- **Problem**: Model may overfit to meta-training tasks- **Symptom**: Great on meta-train, poor on meta-test- **Solution**: Proper train/val/test split at task level### When NOT to use Meta-Learning:❌ **Abundant labeled data** → Use standard supervised learning❌ **Tasks are very different** → Distribution mismatch problem❌ **Single task** → No task distribution to learn from❌ **Extreme computational constraints** → Meta-learning is expensive

<a name='9'></a>## 9 - Course Roadmap: Your Learning JourneyThis course is structured to build your understanding progressively:### 🏗️ **Module 1: Foundations** (Tutorials 01-02)- ✅ **Tutorial 01**: Introduction (you are here!)- 📊 **Tutorial 02**: Learning curves and meta-learning metrics### 🎯 **Module 2: Metric-Based Methods** (Tutorials 03-03c)- 🔵 **Tutorial 03**: Prototypical Networks- 📦 **Tutorial 03b**: Real datasets (Omniglot, Mini-ImageNet)- 🎯 **Tutorial 03c**: Matching Networks### 🔄 **Module 3: Optimization-Based** (Tutorial 04)- 🚀 **Tutorial 04**: MAML (Model-Agnostic Meta-Learning)### 🧠 **Module 4: Advanced Topics** (Tutorials 05-08)- 💾 **Tutorial 05**: Memory-augmented meta-learning- 🤖 **Tutorial 06**: Meta-Reinforcement Learning- 🎨 **Tutorial 07**: Skill discovery and hierarchical RL- 🛡️ **Tutorial 08**: Out-of-distribution generalization### 🎓 **Module 5: Final Project** (Tutorial 09)- 🏆 **Tutorial 09**: Complete end-to-end system### Estimated Time:- **Total**: 20-30 hours- **Per tutorial**: 2-3 hours- **Schedule**: 1-2 tutorials per week = 6-12 weeks### Prerequisites:✅ **Required**:- Python programming- Basic ML (supervised learning)- PyTorch basics- Linear algebra, calculus⚠️ **Recommended**:- Deep learning (CNNs, RNNs)- Reinforcement learning basics (for Tutorials 06-07)- Experience with Jupyter notebooks

---

## 💻 Parte 4: Ejercicio 1 - Modelo Simple de Regresión

Vamos a crear un modelo de red neuronal simple para regresión.

**Tu tarea**: Completa la implementación del modelo.

In [ ]:
class SimpleRegressionModel(nn.Module):
    """
    Red neuronal simple para regresión.
    
    Arquitectura:
        Input (1D) → Hidden (40) → Hidden (40) → Output (1D)
    """
    
    def __init__(self, input_dim=1, hidden_dim=40, output_dim=1):
        super(SimpleRegressionModel, self).__init__()
        
        # TODO: Define las capas de la red
        # Necesitas:
        # 1. Una capa lineal de input_dim a hidden_dim
        # 2. Una capa lineal de hidden_dim a hidden_dim
        # 3. Una capa lineal de hidden_dim a output_dim
        
        self.fc1 = None  # TODO: Reemplaza None con nn.Linear(...)
        self.fc2 = None  # TODO: Reemplaza None con nn.Linear(...)
        self.fc3 = None  # TODO: Reemplaza None con nn.Linear(...)
    
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: Input tensor [batch_size, input_dim]
        
        Returns:
            output: Predicciones [batch_size, output_dim]
        """
        # TODO: Implementa el forward pass
        # Usa activación ReLU entre capas
        # Recuerda: x -> fc1 -> ReLU -> fc2 -> ReLU -> fc3 -> output
        
        pass  # TODO: Elimina esta línea y escribe tu código


# Sistema de pistas
hints_model = HintSystem([
    "Usa nn.Linear(in_features, out_features) para crear capas lineales.",
    "En el forward pass, usa torch.relu() o F.relu() para activaciones.",
    "La estructura es: x = relu(fc1(x)), x = relu(fc2(x)), x = fc3(x)",
    "Solución completa: self.fc1 = nn.Linear(input_dim, hidden_dim), similar para fc2 y fc3"
])

In [ ]:
# Para ver pistas, ejecuta esta celda múltiples veces
hints_model.show_hint()

In [ ]:
# ✅ TEST 1: Verificar que el modelo se construye correctamente

def test_model_construction():
    model = SimpleRegressionModel()
    
    # Verificar que las capas existen
    assert hasattr(model, 'fc1'), "El modelo debe tener una capa fc1"
    assert hasattr(model, 'fc2'), "El modelo debe tener una capa fc2"
    assert hasattr(model, 'fc3'), "El modelo debe tener una capa fc3"
    
    # Verificar que no son None
    assert model.fc1 is not None, "fc1 no debe ser None"
    assert model.fc2 is not None, "fc2 no debe ser None"
    assert model.fc3 is not None, "fc3 no debe ser None"
    
    # Verificar dimensiones
    assert model.fc1.in_features == 1, "fc1 debe tener input_dim=1"
    assert model.fc1.out_features == 40, "fc1 debe tener output=40"
    assert model.fc3.out_features == 1, "fc3 debe tener output_dim=1"
    
    print_success("✅ Modelo construido correctamente!")

run_test(test_model_construction, "Test de Construcción del Modelo")

In [ ]:
# ✅ TEST 2: Verificar que el forward pass funciona

def test_forward_pass():
    model = SimpleRegressionModel()
    
    # Crear input de prueba
    x = torch.randn(5, 1)  # Batch de 5 ejemplos
    
    # Forward pass
    output = model(x)
    
    # Verificar shape
    assert output.shape == (5, 1), f"Output debe tener shape (5, 1), pero tiene {output.shape}"
    
    # Verificar que no hay NaN o Inf
    assert not torch.isnan(output).any(), "Output contiene NaN"
    assert not torch.isinf(output).any(), "Output contiene Inf"
    
    print_success("✅ Forward pass funciona correctamente!")

run_test(test_forward_pass, "Test de Forward Pass")

---

## 💻 Parte 5: Ejercicio 2 - Entrenamiento Simple en Una Tarea

Ahora vamos a entrenar el modelo en una sola tarea usando gradient descent tradicional.

**Tu tarea**: Completa la función de entrenamiento.

In [ ]:
def train_on_task(model, task, n_steps=100, lr=0.01):
    """
    Entrena el modelo en una tarea específica.
    
    Args:
        model: Modelo a entrenar
        task: Diccionario con x_support, y_support
        n_steps: Número de pasos de gradient descent
        lr: Learning rate
    
    Returns:
        losses: Lista de pérdidas durante el entrenamiento
    """
    # TODO: Crea un optimizador SGD con learning rate lr
    optimizer = None  # TODO: Usa optim.SGD(model.parameters(), lr=lr)
    
    # Criterio de pérdida (MSE para regresión)
    criterion = nn.MSELoss()
    
    # Extraer datos de soporte
    x_support = task['x_support']
    y_support = task['y_support']
    
    losses = []
    
    for step in range(n_steps):
        # TODO: Implementa el loop de entrenamiento
        # 1. Zero gradients: optimizer.zero_grad()
        # 2. Forward pass: predictions = model(x_support)
        # 3. Calcular loss: loss = criterion(predictions, y_support)
        # 4. Backward pass: loss.backward()
        # 5. Update parameters: optimizer.step()
        # 6. Guardar loss: losses.append(loss.item())
        
        pass  # TODO: Elimina esta línea y escribe tu código
    
    return losses


# Sistema de pistas
hints_training = HintSystem([
    "El loop de entrenamiento sigue el patrón estándar: zero_grad → forward → loss → backward → step.",
    "optimizer.zero_grad() limpia los gradientes, loss.backward() calcula gradientes, optimizer.step() actualiza parámetros.",
    "No olvides llamar loss.item() para obtener el valor numérico del loss (en lugar del tensor).",
    "Estructura: optimizer.zero_grad(); pred = model(x); loss = criterion(pred, y); loss.backward(); optimizer.step(); losses.append(loss.item())"
])

In [ ]:
# Para ver pistas
hints_training.show_hint()

In [ ]:
# ✅ TEST 3: Verificar que el entrenamiento funciona

def test_training():
    model = SimpleRegressionModel()
    task = create_sine_task(k_shot=20, q_query=10)
    
    losses = train_on_task(model, task, n_steps=50, lr=0.01)
    
    # Verificar que devuelve una lista
    assert isinstance(losses, list), "train_on_task debe devolver una lista"
    assert len(losses) == 50, "Debe haber 50 valores de loss"
    
    # Verificar que el loss disminuye
    assert losses[-1] < losses[0], "El loss debe disminuir durante el entrenamiento"
    
    print_success(f"✅ Entrenamiento funciona! Loss inicial: {losses[0]:.4f}, Loss final: {losses[-1]:.4f}")

run_test(test_training, "Test de Entrenamiento")

---

## 📊 Parte 6: Visualizar Resultados

Ahora vamos a entrenar el modelo y visualizar sus predicciones.

In [ ]:
# Crear una nueva tarea
task = create_sine_task(k_shot=10, q_query=50)

# Crear y entrenar modelo
model = SimpleRegressionModel()
losses = train_on_task(model, task, n_steps=200, lr=0.01)

# Hacer predicciones
model.eval()
with torch.no_grad():
    y_pred = model(task['x_query'])

# Visualizar
plot_few_shot_results(
    task['x_support'], task['y_support'],
    task['x_query'], task['y_query'],
    y_pred=y_pred,
    title="Resultados después de Entrenamiento (200 pasos)"
)

# Plot de la curva de aprendizaje
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Paso de Entrenamiento')
plt.ylabel('Loss (MSE)')
plt.title('Curva de Aprendizaje')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\n📊 Métricas Finales:")
print(f"  Loss inicial: {losses[0]:.4f}")
print(f"  Loss final: {losses[-1]:.4f}")
print(f"  Reducción: {(1 - losses[-1]/losses[0])*100:.1f}%")

---

## 🎯 Parte 7: El Problema del ML Tradicional

El enfoque que acabas de implementar tiene una **limitación crítica**:

### ❌ Problema:
- Necesitas entrenar **desde cero** para cada nueva tarea
- Requieres **muchos pasos de gradient descent**
- No aprovechas el conocimiento de tareas anteriores

### ✅ Solución: Meta-Learning
En los próximos tutoriales aprenderás cómo:
1. Entrenar en múltiples tareas simultáneamente
2. Aprender una **inicialización óptima** que permite adaptación rápida
3. Usar algoritmos como **MAML** que logran adaptación en 1-5 pasos

### Experimento:
Ejecuta la siguiente celda para ver cómo un modelo sin entrenar falla completamente:

In [ ]:
# Modelo sin entrenar (inicialización aleatoria)
untrained_model = SimpleRegressionModel()

# Nueva tarea
new_task = create_sine_task(k_shot=5, q_query=50)

# Predicción sin entrenamiento
untrained_model.eval()
with torch.no_grad():
    untrained_pred = untrained_model(new_task['x_query'])

# Modelo entrenado en la tarea
trained_model = SimpleRegressionModel()
_ = train_on_task(trained_model, new_task, n_steps=100, lr=0.01)
trained_model.eval()
with torch.no_grad():
    trained_pred = trained_model(new_task['x_query'])

# Visualizar comparación
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Sin entrenar
ax1.scatter(new_task['x_support'], new_task['y_support'], c='blue', s=100, label='Support', zorder=3)
ax1.scatter(new_task['x_query'], new_task['y_query'], c='green', s=50, alpha=0.6, label='True Query')
ax1.scatter(new_task['x_query'], untrained_pred, c='red', s=50, alpha=0.6, label='Predicted', marker='^')
ax1.set_title('❌ Modelo Sin Entrenar (Inicialización Aleatoria)', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Entrenado
ax2.scatter(new_task['x_support'], new_task['y_support'], c='blue', s=100, label='Support', zorder=3)
ax2.scatter(new_task['x_query'], new_task['y_query'], c='green', s=50, alpha=0.6, label='True Query')
ax2.scatter(new_task['x_query'], trained_pred, c='red', s=50, alpha=0.6, label='Predicted', marker='^')
ax2.set_title('✅ Modelo Entrenado (100 pasos de GD)', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🎯 Observación clave:")
print("El modelo sin entrenar hace predicciones completamente aleatorias.")
print("Necesitamos 100+ pasos de entrenamiento para cada nueva tarea.")
print("\n💡 En el siguiente tutorial veremos cómo Meta-Learning soluciona esto!")

---

## 🎓 Resumen y Conclusiones

### ✅ Lo que aprendiste:

1. **Meta-Learning** es sobre aprender el proceso de aprendizaje, no solo parámetros
2. Trabajamos con **conjuntos de tareas**, no conjuntos de datos
3. **Support set** = ejemplos para adaptación, **Query set** = evaluación
4. ML tradicional necesita entrenar desde cero para cada tarea

### 🚀 Próximos Pasos:

En el siguiente tutorial (**02_curva_aprendizaje.ipynb**) compararemos:
- ML Tradicional vs Transfer Learning vs Meta-Learning
- Velocidades de adaptación
- Eficiencia en pocos ejemplos

### 📚 Recursos Adicionales:

- Paper: [Learning to Learn](https://link.springer.com/chapter/10.1007/978-1-4615-5529-2_5) - Thrun & Pratt, 1998
- Blog: [Meta-Learning Explained](https://lilianweng.github.io/posts/2018-11-30-meta-learning/)

---

## 🎉 ¡Felicidades!

Has completado el primer tutorial de Meta-Learning. Ahora entiendes los conceptos fundamentales y has implementado tu primer sistema de aprendizaje adaptativo básico.

**🔥 Desafío Opcional:** Intenta modificar la arquitectura del modelo (más capas, más neuronas) y observa cómo afecta el entrenamiento.
